<div style="font-size:30px;font-weight:700;color:#111827;padding-bottom:8px;margin:18px 0;">
LangGraph 첫걸음 — 툴을 골라 쓰는 QA 그래프 만들기
</div>

LangChain의 `prompt | llm | parser` 체인까지 이해한 분을 대상으로,
**LangGraph**를 처음부터 끝까지 한 단계씩 만들어 봅니다.

이 노트북 하나만 위에서 아래로 실행하면 아래 서비스가 완성됩니다.

> 사용자가 질문하면 → LLM이 **도구가 필요한지 스스로 판단**하고 → 필요하면 도구를 실행한 뒤 → 그 결과를 근거로 최종 답변을 만든다.

# STEP 0. 왜 LangGraph인가?

## LangChain 체인의 한계

LangChain의 체인은 **한 방향으로 흐르는 파이프**입니다.

```
prompt ──> llm ──> parser ──> 결과
```

깔끔하지만, 실제 서비스를 만들다 보면 이런 요구가 생깁니다.

| 필요한 것 | 체인으로 가능한가? |
|---|---|
| "이 질문은 도구가 필요하네" → **갈라지기(분기)** | ❌ 어렵다 |
| "답이 부실하네, 다시 시도" → **되돌아가기(반복)** | ❌ 어렵다 |
| 앞 단계의 중간 결과를 뒤 단계가 **공유** | ❌ 매번 수동으로 넘겨야 함 |
| 지금 어느 단계까지 왔는지 **관찰** | ❌ 블랙박스 |

## LangGraph의 해법

LangGraph는 이 흐름을 **그래프**로 다시 정의합니다. 딱 3개 개념뿐입니다.

| 개념 | 한 줄 정의 | 비유 |
|---|---|---|
| **State (상태)** | 모든 단계가 함께 보고 함께 고쳐 쓰는 하나의 딕셔너리 | 팀이 돌려 보는 **공용 노트** |
| **Node (노드)** | State를 받아서, 바뀐 부분만 돌려주는 함수 | 노트에 한 줄 적는 **담당자** |
| **Edge (엣지)** | 다음에 누구에게 노트를 넘길지 정하는 선 | **결재선** |

> 💡 핵심: **노드 하나하나는 그냥 파이썬 함수**입니다. 그 안에서 LangChain 체인을 호출할 뿐입니다.
> 즉 LangGraph는 LangChain을 대체하는 게 아니라, LangChain 체인들을 **엮는 방법**입니다.

## 오늘 만들 그래프

```
                    ┌──────────────┐
       (시작) ─────> │  tool_select │   질문을 보고 도구를 고른다
                    └──────┬───────┘
                           │  ← 조건부 엣지(Router): 도구를 골랐나?
                 ┌─────────┴──────────┐
            YES  │                    │  NO
                 v                    │
          ┌─────────────┐             │
          │  tool_call  │  도구 실행    │
          └──────┬──────┘             │
                 │                    │
                 └────────┬───────────┘
                          v
                   ┌─────────────┐
                   │  question   │   최종 답변 생성
                   └──────┬──────┘
                          v
                        (END)
```

# STEP 1. 설치와 환경 준비

In [ ]:
# %%capture
# %pip install -q -U langgraph langchain-core transformers accelerate bitsandbytes

In [ ]:
import torch

print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"사용 GPU: {torch.cuda.get_device_name(0)}")

# STEP 2. 로컬 모델 로딩

Hugging Face에 올라온 **4bit 양자화 Qwen2.5-Instruct**를 `transformers`로 직접 불러옵니다.

> 💡 **모델 크기 선택**
> - `1.5B` : 도구 선택(JSON 생성) 정확도가 눈에 띄게 좋습니다. **수업용 권장.**
> - `0.5B` : 더 가볍고 빠르지만, JSON 형식을 자주 틀립니다. GPU가 아주 작을 때만 사용하세요.
>
> 이 노트북은 JSON 파싱이 실패해도 그래프가 죽지 않도록 방어 코드를 넣어 두었습니다.

In [ ]:
MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
# MODEL_NAME = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"   # 더 가벼운 대안

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
)

model.eval()
print("모델 로딩 완료:", MODEL_NAME)

# STEP 3. 커스텀 ChatModel 만들기

로컬 모델은 그 자체로는 LangChain이 쓸 수 없습니다.
`BaseChatModel`을 상속해서 **LangChain 규격의 옷을 입혀** 주어야
`invoke()` / `batch()` / `stream()` 과 `|` 파이프 연산을 그대로 쓸 수 있습니다.

구현할 것은 네 가지입니다.

1. `model`, `tokenizer`를 **Pydantic 필드**로 선언 (BaseChatModel이 Pydantic 기반이므로 필수)
2. LangChain 메시지 → Qwen 채팅 형식으로 변환
3. `_generate()` : `invoke()` / `batch()`가 호출
4. `_stream()` : `stream()`이 호출

In [ ]:
import torch
from threading import Thread
from typing import Any, Dict, Iterator, List, Optional
from pydantic import ConfigDict

In [ ]:
from langchain_core.callbacks import CallbackManagerForLLMRun
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, AIMessageChunk, BaseMessage, HumanMessage, SystemMessage
from langchain_core.outputs import ChatGeneration, ChatGenerationChunk, ChatResult

from transformers import TextIteratorStreamer

In [ ]:
class QwenChatModel(BaseChatModel):
    """Qwen2.5-Instruct를 LangChain ChatModel로 감싸는 클래스."""

    # BaseChatModel은 Pydantic 기반이므로 필드 선언이 필요합니다.
    model: Any
    tokenizer: Any

    max_tokens: int = 512
    do_sample: bool = True
    temperature: float = 0.7
    top_p: float = 0.9

    model_config = ConfigDict(arbitrary_types_allowed=True)

    @property
    def _llm_type(self) -> str:
        return "qwen2.5-custom-chatmodel"

    def _tokenize(self, messages: List[BaseMessage]):
        """LangChain 메시지를 Qwen 채팅 형식으로 변환하고 토큰화합니다."""
        chat = []

        for message in messages:
            if isinstance(message, SystemMessage):
                role = "system"
            elif isinstance(message, HumanMessage):
                role = "user"
            elif isinstance(message, AIMessage):
                role = "assistant"
            else:
                role = "user"

            chat.append({"role": role, "content": message.content})

        inputs = self.tokenizer.apply_chat_template(
            chat,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )
        return inputs.to(self.model.device)

    def _generation_options(self, **kwargs):
        """model.generate()에 공통으로 전달할 옵션입니다."""
        do_sample = kwargs.get("do_sample", self.do_sample)

        options = {
            # 주의: max_length(전체 길이)가 아니라 max_new_tokens(새로 만들 길이)를 씁니다.
            "max_length": kwargs.get("max_tokens", self.max_tokens),
            "do_sample": do_sample,
            "pad_token_id": self.tokenizer.pad_token_id or self.tokenizer.eos_token_id,
        }

        if do_sample:
            options["temperature"] = kwargs.get("temperature", self.temperature)
            options["top_p"] = kwargs.get("top_p", self.top_p)

        return options

    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> ChatResult:
        """invoke()와 batch()가 사용하는 메서드입니다."""
        inputs = self._tokenize(messages)
        input_length = inputs["input_ids"].shape[1]

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                **self._generation_options(**kwargs),
            )

        new_tokens = outputs[0][input_length:]
        text = self.tokenizer.decode(
            new_tokens,
            skip_special_tokens=True,
        ).strip()

        return ChatResult(
            generations=[ChatGeneration(message=AIMessage(content=text))]
        )

    def _stream(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> Iterator[ChatGenerationChunk]:
        """stream()이 사용하는 메서드입니다."""
        inputs = self._tokenize(messages)

        streamer = TextIteratorStreamer(
            self.tokenizer,
            skip_prompt=True,
            skip_special_tokens=True,
        )

        thread = Thread(
            target=self.model.generate,
            kwargs={
                **inputs,
                **self._generation_options(**kwargs),
                "streamer": streamer,
            },
        )
        thread.start()

        for text in streamer:
            chunk = ChatGenerationChunk(
                message=AIMessageChunk(content=text)
            )

            if run_manager:
                run_manager.on_llm_new_token(text, chunk=chunk)

            yield chunk

        thread.join()

## 용도가 다르면 설정도 달라야 합니다

같은 모델이라도 **역할에 따라 생성 옵션을 다르게** 주는 것이 좋습니다.

| 인스턴스 | 용도 | 설정 | 이유 |
|---|---|---|---|
| `llm_router` | 도구 선택(JSON 출력) | `do_sample=False` (그리디) | 형식이 정확해야 함. 창의성은 독. |
| `llm_answer` | 최종 답변 생성 | `do_sample=True` | 자연스러운 문장이 필요함. |

In [ ]:
llm_router = QwenChatModel(
    model=model, tokenizer=tokenizer,
    max_tokens=1024,        # JSON 한 줄이면 충분
    do_sample=False,       # 항상 같은 답 → 형식이 안정적
)

llm_answer = QwenChatModel(
    model=model, tokenizer=tokenizer,
    max_tokens=1024,
    do_sample=True, temperature=0.7, top_p=0.9,
)

print("두 개의 ChatModel 준비 완료")

### 동작 확인 (LangChain 복습)

In [ ]:
response = llm_answer.invoke("LangGraph를 한 문장으로 설명해 주세요.")
print(type(response).__name__)   # AIMessage
print(response.content)

In [ ]:
# 스트리밍도 그대로 동작합니다
for chunk in llm_answer.stream("가을에 대한 짧은 시 한 편."):
    print(chunk.content, end="", flush=True)

# STEP 4. 도구(Tool) 정의

에이전트가 쓸 도구를 만듭니다. 도구는 두 벌로 준비합니다.

| 이름 | 정체 | 누가 보는가 |
|---|---|---|
| `TOOL_FUNCS` | **진짜 파이썬 함수** | 파이썬이 실행할 때 |
| `TOOLS` | 함수의 **설명서(JSON 스키마)** | LLM이 "무슨 도구가 있지?" 읽을 때 |

> 💡 LLM은 파이썬 함수를 실행할 수 없습니다. LLM이 하는 일은 오직
> **"이 도구를, 이 인자로 부르세요"라는 텍스트를 만드는 것**뿐이고,
> 실제 실행은 우리 파이썬 코드가 합니다. 이것이 tool calling의 전부입니다.

In [ ]:
def get_weather(city: str) -> str:
    """도시의 현재 날씨를 조회합니다. (데모용 가짜 데이터)"""
    fake_db = {
        "서울": "맑음, 25℃",
        "부산": "흐림, 23℃",
        "대구": "비, 21℃",
    }
    return f"{city}: {fake_db.get(city, '맑음, 25℃')} (데모 데이터)"


def add(a: float, b: float) -> float:
    """두 수를 더합니다."""
    return float(a) + float(b)

In [ ]:
# 1) LLM에게 보여줄 설명서
TOOLS: Dict[str, Dict[str, Any]] = {
    "get_weather": {
        "description": "도시의 현재 날씨를 조회한다",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"],
        },
    },
    "add": {
        "description": "두 수를 더한다",
        "parameters": {
            "type": "object",
            "properties": {"a": {"type": "number"}, "b": {"type": "number"}},
            "required": ["a", "b"],
        },
    },
}

# 2) 파이썬이 실제로 실행할 함수 표
TOOL_FUNCS = {
    "get_weather": get_weather,
    "add": add,
}

print("등록된 도구:", ", ".join(TOOLS.keys()))

In [ ]:
# 도구는 그냥 함수일 뿐입니다. 직접 불러 봅시다.
print(get_weather("서울"))
print(add(2, 3))

# STEP 5. LangChain 체인 2개 만들기

여기까지는 **아직 LangGraph가 아닙니다.** 익숙한 LangChain 체인 그대로입니다.
나중에 이 두 체인이 각각 노드 하나가 됩니다.

| 체인 | 하는 일 | 출력 |
|---|---|---|
| `select_tool_chain` | 질문을 보고 도구를 고른다 | `{"tool": "...", "args": {...}}` |
| `question_chain` | 도구 결과를 근거로 답변한다 | 자연어 문장 |

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
import json, re

## 5-1. 도구 선택 체인

프롬프트에 **"JSON 객체 하나만 출력하라"** 고 아주 강하게 지시합니다.

> `{{` 와 `}}` 는 이중 중괄호입니다. `ChatPromptTemplate`이 `{변수}` 를 치환하기 때문에,
> **진짜 중괄호 문자**를 쓰려면 두 번 겹쳐 써야 합니다.

In [ ]:
## 치환될 변수 : tool_names, tool_schema, input
select_instruct = """\
당신은 도구 라우터입니다. 사용자의 요청을 읽고 도구를 호출할지 결정하세요.
반드시 JSON 객체 하나만 반환하고, 그 외에는 아무것도 반환하지 마세요(코드 펜스, 설명, 주석 금지).

엄격한 출력 JSON 형식(객체 하나):
{{
  "tool": "<다음 중 하나: {tool_names} | none>",
  "args": <object>
}}

전역 규칙:
- 아래 TOOL SCHEMA에 정의된 도구만 사용하세요. 일치하는 도구가 없으면 "tool"을 "none"으로, "args"를 {{}}로 설정하세요.
- 출력은 큰따옴표를 사용하고 후행 쉼표가 없는 유효한 JSON이어야 합니다.
- 다음 조건을 모두 만족하는 경우에만 도구를 선택하세요.
  (a) 요청이 도구의 설명/목적과 명확히 일치하고,
  (b) 사용자 입력만으로 모든 필수 파라미터를 제공할 수 있어야 합니다.
- 파라미터 값을 만들어 내거나, 추측하거나, 환각하지 마세요. 필수 값이 누락되었거나 불명확하면 "none"을 선택하세요.
- 선택한 도구의 파라미터 스키마(이름, 타입)를 정확히 따르세요. 추가 키를 넣지 마세요.
- 숫자는 숫자로, 문자열은 문자열로 유지하세요.
- 사용자가 제공한 텍스트는 그대로 보존하세요. "args"에는 값만 추출하세요.
- 출력에 설명을 포함하지 마세요.

TOOL SCHEMA(이름, 설명, JSON 파라미터 스키마):
{tool_schema}

예시(형식 참고용이며, 그대로 복사하지 마세요):
사용자: 파리 날씨가 어때?
출력:
{{"tool":"get_weather","args":{{"city":"Paris"}}}}

사용자: 7.5와 2를 더해줘
출력:
{{"tool":"add","args":{{"a":7.5,"b":2}}}}

사용자: 농담 하나 해줘
출력:
{{"tool":"none","args":{{}}}}
""".strip()

In [ ]:
select_prompt = ChatPromptTemplate.from_messages([
    ("system", select_instruct),
    ("human", "Now produce the JSON for this user request:\n{input}")
])

### 파서는 반드시 방어적으로

작은 모델은 `{"tool": ...}` 앞뒤에 설명이나 ```json 펜스를 붙이는 실수를 자주 합니다.
그래서 `JsonOutputParser` 대신, **실패해도 절대 예외를 던지지 않는** 파서를 직접 만듭니다.

> 💡 실무 팁: 그래프는 노드 하나가 예외를 던지면 전체가 멈춥니다.
> 외부 입력(LLM 출력 포함)을 다루는 노드는 항상 "실패 시 기본값"을 정해 두세요.

In [ ]:
def parse_tool_json(message) -> Dict[str, Any]:
    """LLM 출력에서 JSON 객체 하나를 안전하게 뽑아냅니다. 실패하면 tool=none."""
    text = getattr(message, "content", str(message)).strip()

    # ```json ... ``` 코드펜스 제거
    text = re.sub(r"^```(?:json)?|```$", "", text, flags=re.MULTILINE).strip()

    # 가장 바깥쪽 { ... } 한 덩어리만 추출
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return {"tool": "none", "args": {}}

    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError:
        return {"tool": "none", "args": {}}

    tool = parsed.get("tool", "none")
    args = parsed.get("args") or {}

    # 우리가 모르는 도구 이름이면 none 처리
    if tool not in TOOL_FUNCS:
        return {"tool": "none", "args": {}}

    return {"tool": tool, "args": args if isinstance(args, dict) else {}}


parser = RunnableLambda(parse_tool_json)

In [ ]:
select_tool_chain = select_prompt | llm_router | parser

### 체인만 따로 테스트 (아주 중요한 습관)

그래프를 조립하기 **전에** 체인 하나하나가 제대로 도는지 확인해야
나중에 문제가 생겼을 때 어디가 잘못됐는지 바로 알 수 있습니다.

In [ ]:
tool_schema_text = json.dumps(TOOLS, ensure_ascii=False, indent=2)
tool_names_text = ", ".join(TOOLS.keys())

for q in ["서울 날씨 어때?", "2와 3을 더한 결과는?", "좋은 회의 아이스브레이커 알려줘"]:
    result = select_tool_chain.invoke({
        "input": q,
        "tool_schema": tool_schema_text,
        "tool_names": tool_names_text,
    })
    print(f"{q:30s} -> {result}")

## 5-2. 답변 생성 체인

도구 결과(`observation`)가 있으면 **그것만 근거로**, 없으면 모델 지식으로 답하게 합니다.

In [ ]:
question_instruct = """\
반드시 간결하고 정확하게 한국어로 답변하세요.

도구 결과:
{observation}

지침:
- 도구 결과가 비어 있지 않다면, 그 결과에 근거한 짧은 문단 하나만 작성하세요. 결과와 모순되는 내용을 작성하지 마세요.
- 도구 결과가 비어 있다면, 당신의 지식으로 직접 답변하세요. 모르는 경우에는 "잘 모르겠습니다."라고 답하세요. 사실을 지어내지 마세요.
- 도구, 파이프라인, 또는 추론 과정을 언급하지 마세요. 최종 답변만 출력하세요.
- 약 1~3문장으로 간결하게 작성하세요.
""".strip()

question_prompt = ChatPromptTemplate.from_messages([
    ("system", question_instruct),
    ("human", "사용자 질문:\n{input}")
])

question_chain = question_prompt | llm_answer

In [ ]:
# 도구 결과가 있을 때
print(question_chain.invoke({
    "input": "서울 날씨 어때?",
    "observation": "서울: 맑음, 25℃ (데모 데이터)",
}).content)

print("-" * 60)

# 도구 결과가 없을 때
print(question_chain.invoke({
    "input": "좋은 회의 아이스브레이커 알려줘",
    "observation": "",
}).content)

> 🔎 **여기서 잠깐.**
> 두 체인은 잘 돕니다. 그런데 "언제 첫 번째를 쓰고, 언제 두 번째로 넘어갈지"는
> 지금 아무도 결정해 주지 않습니다. 우리가 `if` 문으로 직접 짜야 하죠.
>
> **바로 이 '흐름 제어'를 코드가 아니라 그림(그래프)으로 선언하는 것이 LangGraph입니다.**
> 이제부터가 진짜 LangGraph입니다.

# STEP 6. State — 그래프의 공용 노트

LangGraph의 첫 번째 개념입니다.

State는 **모든 노드가 함께 보고 함께 채워 나가는 하나의 딕셔너리**입니다.
`TypedDict`로 "이 노트에는 어떤 칸이 있다"를 미리 선언합니다.

| 칸 | 누가 채우나 | 내용 |
|---|---|---|
| `input` | 사용자 | 원래 질문 |
| `tool_schema`, `tool_names` | 사용자 | 도구 설명서 |
| `selection` | `tool_select` 노드 | 고른 도구와 인자 |
| `observation` | `tool_call` 노드 | 도구 실행 결과 |
| `answer` | `question` 노드 | 최종 답변 |

In [ ]:
from typing import TypedDict, Optional

class StateToolQA(TypedDict):
    input: str                       # 사용자 질문
    tool_schema: str                 # 도구 설명서(JSON 문자열)
    tool_names: str                  # 도구 이름 목록
    selection: Dict[str, Any]        # {"tool": ..., "args": {...}}
    observation: Optional[Any]       # 도구 실행 결과
    answer: Optional[str]            # 최종 답변

# STEP 7. Node — State를 조금씩 채우는 함수들

노드의 규칙은 딱 두 줄입니다.

```python
def 노드이름(state) -> dict:
    ...
    return {"바뀐칸": 값}     # 바뀐 칸만 돌려주면 LangGraph가 알아서 합쳐 준다
```

> 💡 `return {**state, ...}` 처럼 전체를 돌려줘도 동작하지만,
> **바뀐 키만 돌려주는 것이 권장 방식**입니다. 누가 무엇을 바꿨는지 한눈에 보이니까요.

## 7-1. `tool_select` 노드 — 도구를 고른다

In [ ]:
def node_tool_select(state: StateToolQA) -> Dict[str, Any]:
    print(f"[node] tool_select  <- input={state['input']!r}")

    selection = select_tool_chain.invoke({
        "input": state["input"],
        "tool_schema": state["tool_schema"],
        "tool_names": state["tool_names"],
    })

    print(f"[node] tool_select  -> selection={selection}")
    return {"selection": selection}

### 노드를 그래프 없이 단독 테스트

노드는 그냥 함수이므로 **딕셔너리를 손으로 만들어 바로 호출**할 수 있습니다.
디버깅할 때 가장 유용한 방법입니다.

In [ ]:
state = {
    "input": "서울 날씨 어때?",
    "tool_schema": tool_schema_text,
    "tool_names": tool_names_text,
}

update = node_tool_select(state)
state = {**state, **update}      # LangGraph가 내부에서 해 주는 병합을 손으로 흉내
state

## 7-2. `tool_call` 노드 — 도구를 실행한다

이 노드에는 LLM이 전혀 등장하지 않습니다. **순수한 파이썬 실행**입니다.

In [ ]:
def node_tool_call(state: StateToolQA) -> Dict[str, Any]:
    selection = state.get("selection") or {}
    tool = selection.get("tool", "none")
    args = selection.get("args") or {}

    print(f"[node] tool_call    <- tool={tool}, args={args}")

    if tool not in TOOL_FUNCS:
        return {"observation": None}

    try:
        result = TOOL_FUNCS[tool](**args)
    except Exception as e:
        # 인자 이름이 틀리거나 타입이 안 맞을 수 있습니다. 그래프를 죽이지 않습니다.
        result = f"TOOL_ERROR: {e}"

    print(f"[node] tool_call    -> observation={result!r}")
    return {"observation": result}

In [ ]:
update = node_tool_call(state)
state = {**state, **update}
state

## 7-3. `question` 노드 — 최종 답변을 만든다

In [ ]:
def node_question(state: StateToolQA) -> Dict[str, Any]:
    observation = state.get("observation")
    print(f"[node] question     <- observation={observation!r}")

    message = question_chain.invoke({
        "input": state["input"],
        # None을 그대로 넣으면 프롬프트에 "None" 이라는 글자가 박힙니다. 빈 문자열로 바꿔 줍니다.
        "observation": observation if observation else "",
    })

    answer = getattr(message, "content", str(message))
    print(f"[node] question     -> answer 생성 완료")
    return {"answer": answer}

In [ ]:
update = node_question(state)
state = {**state, **update}

print("-" * 60)
print(state["answer"])

> ✅ 여기까지 세 노드가 각각 잘 동작합니다.
> 하지만 지금은 **우리가 손으로 순서를 정해서** 하나씩 호출했습니다.
> 이제 이 순서를 LangGraph에게 맡깁니다.

# STEP 8. Router — 갈림길을 정하는 함수

LangGraph에서 **조건부 엣지(conditional edge)** 는 이렇게 생겼습니다.

```python
def 라우터(state) -> str:
    return "다음에_갈_노드_이름"
```

State를 보고 **문자열 하나(다음 노드 이름)** 를 돌려주면 끝입니다.

우리 규칙: 도구를 골랐으면 → `tool_call`, 아니면 → 바로 `question`.

In [ ]:
def after_tool_select(state: StateToolQA) -> str:
    selection = state.get("selection") or {}
    tool = selection.get("tool", "none")

    next_node = "tool_call" if tool in TOOL_FUNCS else "question"
    print(f"[router] tool={tool} -> 다음 노드: {next_node}")
    return next_node

In [ ]:
# 라우터도 단독 테스트가 됩니다
print(after_tool_select({"selection": {"tool": "get_weather", "args": {"city": "서울"}}}))
print(after_tool_select({"selection": {"tool": "none", "args": {}}}))

# STEP 9. 그래프 조립하기

드디어 부품을 붙입니다. 사용하는 함수는 5개뿐입니다.

| 함수 | 하는 일 |
|---|---|
| `StateGraph(StateToolQA)` | 어떤 State를 쓸지 정하며 그래프 시작 |
| `add_node(이름, 함수)` | 노드 등록 |
| `set_entry_point(이름)` | 시작 노드 지정 |
| `add_edge(A, B)` | **무조건** A 다음 B |
| `add_conditional_edges(A, 라우터, 매핑)` | 라우터 결과에 따라 갈라짐 |
| `compile()` | 실행 가능한 앱으로 굳히기 |

In [ ]:
from langgraph.graph import StateGraph, END

## 9-1. 노드 등록

In [ ]:
def build_graph() -> StateGraph:
    graph = StateGraph(StateToolQA)

    graph.add_node("tool_select", node_tool_select)
    graph.add_node("tool_call",   node_tool_call)
    graph.add_node("question",    node_question)

    return graph

## 9-2. 엣지 연결 후 컴파일

`END`는 LangGraph가 미리 만들어 둔 **종료 지점**입니다.
`question` 노드 뒤에 `END`를 붙여야 그래프가 "여기서 끝"이라는 걸 압니다.

In [ ]:
def build_app():
    graph = build_graph()

    # 1) 어디서 시작할까?
    graph.set_entry_point("tool_select")

    # 2) tool_select 다음은 라우터가 결정 (조건부 엣지)
    graph.add_conditional_edges(
        "tool_select",          # 이 노드가 끝나면
        after_tool_select,      # 이 함수를 불러서
        {                       # 반환값을 실제 노드 이름으로 매핑
            "tool_call": "tool_call",
            "question":  "question",
        },
    )

    # 3) tool_call 다음은 무조건 question (일반 엣지)
    graph.add_edge("tool_call", "question")

    # 4) question 다음은 끝
    graph.add_edge("question", END)

    return graph.compile()


app = build_app()
print("그래프 컴파일 완료")

## 9-3. 그래프 그림으로 확인하기

내가 그린 흐름이 맞는지 **눈으로 검증**하는 단계입니다.
환경에 따라 이미지 렌더링이 안 될 수 있어서 3단계로 시도합니다.

In [ ]:
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e1:
    print(f"(이미지 렌더 실패: {type(e1).__name__}) 텍스트로 대체합니다.\n")
    try:
        print(app.get_graph().draw_ascii())
    except Exception:
        print(app.get_graph().draw_mermaid())

# STEP 10. 실행하기

`app.invoke(초기_State)` 한 번이면 그래프 전체가 돕니다.
**초기 State에는 "사용자가 알려줘야 하는 칸"만 넣으면 됩니다.**
나머지 칸(`selection`, `observation`, `answer`)은 노드들이 채웁니다.

In [ ]:
def question_answer(user_input: str) -> str:
    init: StateToolQA = {
        "input": user_input,
        "tool_schema": tool_schema_text,
        "tool_names": tool_names_text,
    }
    out = app.invoke(init)
    return out.get("answer")

## 케이스 1. 날씨 도구가 필요한 질문
`tool_select` → `tool_call` → `question` 경로를 탑니다.

In [ ]:
answer = question_answer("서울 날씨 어때?")
print(f"\n{'='*60}\n{answer}")

## 케이스 2. 계산 도구가 필요한 질문

In [ ]:
answer = question_answer("2와 3을 더한 결과는?")
print(f"\n{'='*60}\n{answer}")

## 케이스 3. 도구가 필요 없는 질문
`tool_select` → (`tool_call` 건너뜀) → `question` 경로를 탑니다.
로그에서 `[node] tool_call` 이 **찍히지 않는 것**을 확인해 보세요. 분기가 작동한 증거입니다.

In [ ]:
answer = question_answer("좋은 회의 아이스브레이커 알려줘")
print(f"\n{'='*60}\n{answer}")

# STEP 11. stream — 노드가 하나씩 끝날 때마다 들여다보기

`invoke()`는 최종 결과만 줍니다. `stream()`을 쓰면 **노드가 하나 끝날 때마다**
그 노드가 무엇을 State에 썼는지 실시간으로 받아볼 수 있습니다.

실제 서비스에서 "지금 검색 중입니다…" 같은 진행 표시를 만들 때 쓰는 방법입니다.

In [ ]:
init: StateToolQA = {
    "input": "부산 날씨 알려줘",
    "tool_schema": tool_schema_text,
    "tool_names": tool_names_text,
}

for step in app.stream(init):
    for node_name, update in step.items():
        print(f"\n🟢 [{node_name}] 완료 — State에 기록한 내용:")
        for key, value in update.items():
            preview = str(value)
            if len(preview) > 120:
                preview = preview[:120] + " ..."
            print(f"    {key} = {preview}")